# asyncio - await专题

## await 的本质含义
在异步编程中，await 并不是一个简单的“等待”动作，而是一个“出让执行权”的信号。它告诉程序的调度员（事件循环）：我现在需要等一个结果，暂时用不到 CPU，你可以先把 CPU 拿走去处理清单上其他 **已经报了名** 的任务。等我等的东西好了，你再回来叫醒我，从这行代码继续往下跑。

## 关键辨析：哪些是“提交了任务”，哪些不是？
这是初学者最容易产生误解的地方，理解这一点就能区分“并发”和“串行”。
- 使用 await 后面接函数调用（例如：await func()）：
这是“立即提交并原地死等”。你把报案单交给了警察，但你搬了个小板凳坐在警察局门口，不处理完你的案子你就不走。虽然你让出了 CPU，但因为你没有提前提交后续的其他任务，CPU 往往只能闲着。
- 使用 asyncio.create_task(func())：
这是真正的“后台提交”。这行代码跑完的瞬间，任务就正式进入了事件循环的待办清单。即使你还没写 await，这个任务其实已经开始在后台计时或排队跑了。
- 使用 asyncio.gather(func1(), func2())：
这是“批量提交”。它瞬间把多个报案单一起塞进警察局，然后返回一个总的凭证。你再 await 这个总凭证，就能实现多个任务同时跑。
- **大坑：直接调用异步函数（例如：func()）：**
这不是提交任务。这只是创建了一个“协程对象”，它就像一张填好的报案单，但你还把它揣在兜里，没有交给警察局。此时代码不会运行。

## await 的使用方法
- 必须在异步环境中使用：
await 关键字只能出现在被 async def 定义的函数（协程）内部。在普通函数里写 await 会直接报语法错误。
- 后面必须接“可等待对象”：
await 后面通常接三种东西：直接调用异步函数产生的协程对象、由 create_task 创建的任务对象、或者是 Ray 的 ObjectRef（对象引用）。如果你尝试 await 一个普通的整数或字符串，程序会崩溃。
- 链式调用的规则：
你可以 await 一个返回协程的函数，也可以 await 一个变量，只要这个变量指向的是上述的可等待对象。

## 核心注意事项（避坑指南）
- 防止“毒化”循环：
千万不要在 await 附近写 time.sleep()。await 是谦让 CPU，而 time.sleep 是霸占 CPU 睡觉。一旦霸占，整个事件循环就瘫痪了，所有后台任务都会跟着一起卡死。
- 避免 CPU 密集型长计算：
如果一行代码不带 await，且需要跑好几秒（比如一个几亿次的循环），它同样会卡死事件循环。这种活儿应该交给 Ray 的远程任务去做，而不是在 asyncio 里硬扛。
- 异常处理的滞后性：
如果你 create_task 提交了一个任务但没有 await 它，这个任务如果报错了，你的主程序可能完全不知道，直到程序结束才弹出一个隐晦的警告。建议重要的任务最后都要 await 一下，或者通过回调来捕捉错误。
- 串行陷阱：
如果你有多个互不相关的任务，千万不要写成一排 await。记住：先用 create_task 提交，或者用 gather 打包，最后再 await。

# 相关扩展与进阶内容
- 超时控制：
可以使用 asyncio.wait_for 来包裹一个协程，并设置一个时间上限。如果 await 的时间超过了限制，它会自动抛出超时异常并尝试取消掉那个任务，这是保护系统响应速度的重要手段。
- 现代化的任务组（TaskGroup）：
在 Python 3.11 之后，官方推荐使用 async with asyncio.TaskGroup() 这种写法。它比 gather 更安全，如果组内一个任务崩了，它会自动确保其他任务也被正确关闭，避免产生“孤儿任务”。
- 迭代器与上下文：
除了 await 函数，还有异步循环（async for）和异步上下文管理器（async with）。它们本质上是在进入循环或进入代码块时，内部执行了 await 动作。
- 与 Ray 的联动：
Ray 的设计非常精妙，它让分布在几百台机器上的任务返回的“取货码”也能被 await。当你 await 一个 Ray 的引用时，你其实是在单机异步的基础上，实现了跨机器的非阻塞等待。

总结一句话：先利用 create_task 填满你的“待办清单”，再通过 await 优雅地交出 CPU 权限，这才是高性能异步程序的正确打开方式。